# خُلاصة: دفتر واجهة التلخيص التاريخية

يوثق الدفتر واجهة التلخيص السابقة باستخدام النموذج المضبوط لمدة 1200 خطوة. لا يُعاد التدريب ولا تُعدّل الأوزان. أُزيل مسار Drive الخاص من الإعدادات المنشورة. يعتمد التشغيل الحالي الذي يجمع التلخيص واسترجاع تقارير التدقيق على [Khulasa_Final_Demo.ipynb](Khulasa_Final_Demo.ipynb)، وليس الواجهة المضمّنة هنا.


## 1. الإعدادات وتثبيت المكتبات


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Avoid importing the preinstalled TensorFlow and its protobuf dependencies.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

# Existing Drive export from the completed 1200-step training run.
MODEL_DIR = Path("/content/drive/MyDrive/AUTHORIZED_PROJECT")
SOURCE_LENGTH = 1024
SOURCE_COMMIT = "d1b24cbeef87aa61e45693fd5f7895a9f8db6f05"  # Stable project version.

# Do NOT reinstall torch; preserve Colab's CUDA-enabled installation.
packages = [
    "transformers==4.57.1",
    "accelerate==1.11.0",
    "sentencepiece==0.2.1",
    "protobuf==6.31.1",
    "safetensors==0.6.2",
    "gradio==5.49.1",
    "trafilatura==2.0.0",
    "urllib3==2.5.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed. No training packages or model weights were changed.")

## 2. ربط التخزين وفحص الملفات


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
if not MODEL_DIR.is_dir():
    raise FileNotFoundError(
        f"لم يُعثر على النموذج: {MODEL_DIR}\n"
        "افتحي Google Drive وتحققي من مسار المجلد الذي انتهى بـ /model، ثم عدّلي MODEL_DIR في الخلية الأولى."
    )
expected = ["config.json", "tokenizer_config.json"]
missing = [name for name in expected if not (MODEL_DIR / name).is_file()]
if missing:
    raise RuntimeError(f"Model export appears incomplete, missing: {missing}")
weights = list(MODEL_DIR.glob("*.safetensors"))
if not weights:
    raise RuntimeError("No safetensors weights found. Do not use the old checkpoint directory.")
print("Model export found:", MODEL_DIR)
print("Weights:", [(x.name, round(x.stat().st_size / 1e9, 3)) for x in weights])
print("No file will be deleted or overwritten.")

## 3. مصدر التطبيق التاريخي


In [ ]:
repo = Path("/content/ArabicNewsGradioApp")
if not (repo / ".git").exists():
    subprocess.run([
        "git", "clone", "--quiet", "--no-checkout",
        "https://github.com/nada-ali-13/Arabic-news-summarizer.git", str(repo)
    ], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "--quiet", "origin", SOURCE_COMMIT], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
if not (repo / "app.py").is_file():
    raise RuntimeError("The pinned repository commit does not contain app.py")
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo))
print("Loaded pinned app source:", SOURCE_COMMIT[:12])

## 4. تحميل النموذج واختبار توليد


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from arabic_summarizer.inference import Summarizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("تحذير: لا يوجد GPU؛ التوليد قد يكون بطيئًا. يفضّل اختيار T4 GPU.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

# FP32 is safe for the saved model; unlike training this is inference only.
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=False, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    str(MODEL_DIR), dtype=torch.float32, use_safetensors=True,
    weights_only=True, low_cpu_mem_usage=True, local_files_only=True,
).to(DEVICE).eval()
model.config.use_cache = True

summarizer = Summarizer(
    model_path=str(MODEL_DIR), device=DEVICE, source_length=SOURCE_LENGTH,
    model=model, tokenizer=tokenizer,
)
SAMPLE_ARTICLE = (
    "أعلنت جامعة المدينة اليوم افتتاح مكتبة جديدة لخدمة الطلاب والباحثين، "
    "وتضم المكتبة قاعات للمطالعة ومختبرًا رقميًا ومجموعة من الكتب العلمية. "
    "وأوضح مدير الجامعة أن المكتبة ستفتح أبوابها الأسبوع المقبل، "
    "وأنها ستوفر دورات تدريبية على البحث في المصادر الإلكترونية، "
    "كما ستتيح للطلاب الاستفادة من الخدمات خلال ساعات الدوام الجامعي."
)
result = summarizer.summarize(SAMPLE_ARTICLE, preference="متوسط")
assert result["summary"].strip(), "Model generated an empty summary"
print("✅ النموذج جاهز؛ الملخص التجريبي:")
print(result["summary"])
print("زمن التوليد:", result["seconds"], "ثانية")

## 5. واجهة Gradio التاريخية

تدعم الواجهة إدخال نص أو رابط مقال عام، مع اختيار طول الملخص. قد يتعذر استخراج صفحات محمية أو ديناميكية. رابط `share=True` عام ومؤقت، ويتوقف بانتهاء جلسة Colab.


In [ ]:
import gradio as gr
from html import escape
from arabic_summarizer.articles import extract_article, ArticleError

assert "summarizer" in globals(), "شغّلي خلايا تحميل النموذج أولًا."

CUSTOM_CSS = """
.gradio-container {direction:rtl; font-family:'Segoe UI',Tahoma,Arial,sans-serif!important;
  max-width:1200px!important; margin:auto!important; background:#f8f9fe;}
body {background:#f8f9fe!important;}
#hero {background:linear-gradient(125deg,#172554,#4338ca,#6d5dfc);
  border-radius:22px; padding:42px 35px; color:white; text-align:center;
  margin-bottom:24px; box-shadow:0 12px 30px rgba(67,56,202,.15);}
#hero h1 {font-size:40px; font-weight:800; color:white; margin:4px 0 8px;}
#hero h2 {font-size:25px; font-weight:650; color:white; margin:10px 0 18px;}
#hero p {max-width:700px;margin:0 auto;color:#e0e7ff;font-size:17px;line-height:2;}
#hero .chip {display:inline-block;margin-bottom:12px;padding:7px 16px;
 background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.25);
 border-radius:50px;font-size:13px;color:white;}
#input-card,#output-card {background:white;border:1px solid #e9eaf4;
 border-radius:20px;padding:23px;box-shadow:0 5px 22px rgba(26,35,88,.045);}
.section-title {font-size:19px;font-weight:750;color:#18244a;margin-bottom:7px;}
.section-subtitle {color:#7b849d;font-size:13px;margin-bottom:20px;}
#article-box textarea,#url-box input,#summary-box textarea {direction:rtl!important;
 text-align:right!important;line-height:1.9!important;font-size:15px!important;
 border-radius:12px!important;}
#summary-box textarea {font-size:17px!important;color:#192653!important;background:#fafbff!important;}
#submit-btn {background:linear-gradient(100deg,#4338ca,#7064fa)!important;
 color:white!important;border:none!important;border-radius:12px!important;
 font-weight:750!important;min-height:52px!important;font-size:17px!important;}
#clear-btn {border-radius:12px!important;min-height:45px!important;}
.metric-grid {display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:12px;}
.metric-item {text-align:center;background:#f4f5ff;padding:17px 8px;border-radius:12px;border:1px solid #e9eaff;}
.metric-number {font-weight:800;color:#4338ca;font-size:21px;}
.metric-label {color:#77809b;font-size:12px;margin-top:5px;}
.footer-note {text-align:center;color:#838ba3;font-size:13px;padding:20px 10px 4px;line-height:1.9;}
.author-note {text-align:center;padding:0 10px 20px;font-size:13px;color:#64748b;}
"""

def summarize_article(text, url, length):
    try:
        text = (text or "").strip()
        url = (url or "").strip()
        if text and url:
            raise ValueError("أدخلي نص المقال أو رابطه، وليس الاثنين معًا.")
        if not text and not url:
            raise ValueError("يرجى إدخال نص مقال أو رابط أولًا.")
        article = extract_article(url) if url else text
        result = summarizer.summarize(article, preference=length)
        stats = f"""<div class="metric-grid">
          <div class="metric-item"><div class="metric-number">{int(result['original_words'])}</div><div class="metric-label">كلمات المقال</div></div>
          <div class="metric-item"><div class="metric-number">{int(result['summary_words'])}</div><div class="metric-label">كلمات الملخص</div></div>
          <div class="metric-item"><div class="metric-number">{float(result['compression_percent']):.1f}%</div><div class="metric-label">نسبة الاختصار</div></div>
        </div><p style="color:#8b91a6;font-size:12px;margin-top:15px">زمن التوليد: {float(result['seconds']):.2f} ثانية</p>"""
        return result['summary'], stats, escape(result.get('warning', ''))
    except (ValueError, ArticleError) as exc:
        raise gr.Error(str(exc)) from exc
    except Exception as exc:
        print('Summarization error:',repr(exc))
        raise gr.Error('تعذّر التلخيص. تحققي من المقال أو الرابط وحاولي مجددًا.') from exc

with gr.Blocks(theme=gr.themes.Soft(primary_hue='indigo',neutral_hue='slate'),
               css=CUSTOM_CSS,title='خُلاصة | ملخص الأخبار الذكي') as demo:
    gr.HTML('''<div id="hero"><div class="chip">✦ اقرأ أقل، واعرف أكثر</div>
      <h1>خُلاصة</h1><h2>الخبر كاملًا؟ خذ خُلاصته.</h2>
      <p>أخبار كثيرة، ووقت قليل؟ الصق المقال أو أضف رابطه، ودع خُلاصة يحوّله إلى ملخص عربي سريع وواضح، بالطول الذي يناسبك.</p></div>''')
    with gr.Row(equal_height=False):
        with gr.Column(scale=1,elem_id='input-card'):
            gr.HTML('<div class="section-title">إدخال المقال</div><div class="section-subtitle">الصق نص الخبر أو أدخل رابطه لتلخيصه.</div>')
            article_input=gr.Textbox(label='نص المقال',placeholder='اكتب أو الصق المقال الإخباري هنا...',lines=12,max_lines=18,elem_id='article-box')
            url_input=gr.Textbox(label='رابط المقال',placeholder='https://example.com/news',elem_id='url-box')
            length_input=gr.Radio(['قصير','متوسط','طويل'],value='متوسط',label='طول الملخص')
            submit_btn=gr.Button('تلخيص المقال',variant='primary',elem_id='submit-btn')
            clear_btn=gr.Button('مسح الحقول',variant='secondary',elem_id='clear-btn')
        with gr.Column(scale=1,elem_id='output-card'):
            gr.HTML('<div class="section-title">نتيجة التلخيص</div><div class="section-subtitle">سيظهر الملخص الناتج هنا بعد المعالجة.</div>')
            summary_output=gr.Textbox(label='الملخص',placeholder='الملخص سيظهر هنا...',lines=16,interactive=False,show_copy_button=True,elem_id='summary-box')
            stats_output=gr.HTML()
            warning_output=gr.Markdown()
    submit_btn.click(fn=summarize_article,inputs=[article_input,url_input,length_input],outputs=[summary_output,stats_output,warning_output],concurrency_limit=1)
    clear_btn.click(fn=lambda: ('','','','',''),inputs=[],outputs=[article_input,url_input,summary_output,stats_output,warning_output],queue=False)
    gr.HTML('<div class="footer-note">يعتمد التطبيق على نموذج AraT5v2 بعد إعادة تدريبه على الأخبار العربية.<p>قد تحتوي الملخصات الآلية على أخطاء واقعية؛ يُنصح بمراجعة المصدر.</p></div>')
    gr.HTML('<div class="author-note">تصميم وتطوير: <strong style="color:#6366f1;">ندى علي</strong> ✦ © 2026</div>')

demo.queue(max_size=8)
demo.launch(share=True,inline=True)


## نطاق الدفتر وقيوده

النموذج هو AraT5v2-base-1024 بعد الضبط على Arabic XL-Sum. النتائج الموسعة الموثقة على 300 مقال test هي ROUGE-1 = 29.32 وROUGE-2 = 13.89 وROUGE-Lsum = 25.24؛ ليست Accuracy. قد تتضمن الملخصات أخطاء وقائع أو حذفًا. لا تمثل المشاركة المؤقتة استضافة دائمة.
